# PYTORCH RNN trên sp500

Nguyễn Ngọc Hoàng Nam — B23DCCN585 | ASG 06

Các cell nhỏ được sắp theo thứ tự phụ thuộc. Huấn luyện đầy đủ mặc định tắt; số liệu chạy thử không phải kết quả test chính thức.

In [1]:
from pathlib import Path
import sys, json, os
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks": ROOT=ROOT.parent
assert (ROOT/"src").is_dir()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.paths import DATA
from src.io import read_json
import numpy as np
import pandas as pd
from IPython.display import display, Image
print("Project:", ROOT)
print("Data:", DATA)


Project: E:\PTHTTM\ASG_06
Data: E:\PTHTTM\ASG_06_data


## 1. Dữ liệu và hình dạng

In [2]:
from src.data import load_data,split_ids,make_batch
data=load_data('sp500')
x,lengths,y=make_batch(data,split_ids(data,'train')[:8])
print(x.shape,lengths,y)

(8, 30, 7) [30 30 30 30 30 30 30 30] [ 0.01663691  0.00643549 -0.00285511 -0.02509179 -0.00661848 -0.00344913
  0.02172672  0.00313442]


## 2. Hàm xây dựng RNN cơ bản

In [3]:
from src.backends import pytorch_runtime
torch,device=pytorch_runtime(42,cpu=True)
print(torch.__version__,device)

2.5.1+cu121 cpu


In [4]:
def build_pytorch(torch, input_dim, hidden, output_dim):
    class SimpleRNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.rnn=torch.nn.RNN(input_dim,hidden,batch_first=True,nonlinearity="tanh")
            self.rnn.bias_hh_l0.requires_grad_(False)
            self.head=torch.nn.Linear(hidden,output_dim)
        def forward(self,x,lengths):
            packed=torch.nn.utils.rnn.pack_padded_sequence(
                x,lengths.cpu(),batch_first=True,enforce_sorted=False)
            _,state=self.rnn(packed)
            return self.head(state[-1])
    return SimpleRNN()

In [5]:
model=build_pytorch(torch,data['metadata']['input_dim'],32,1)
print(model)

SimpleRNN(
  (rnn): RNN(7, 32, batch_first=True)
  (head): Linear(in_features=32, out_features=1, bias=True)
)


## 3. Lan truyền thuận và hàm mất mát

In [6]:
tensor=torch.tensor(x)
output=model(tensor,torch.tensor(lengths))
print('Output:',output.shape)

Output: torch.Size([8, 1])


In [7]:
target=torch.tensor(y/data['metadata']['target_scale'],dtype=torch.float32)
loss=torch.mean((output[:,0]-target)**2)
loss.backward()
print('Loss minh họa:',float(loss))
print('Norm gradient:',float(model.rnn.weight_ih_l0.grad.norm()))

Loss minh họa: 0.7417923212051392
Norm gradient: 0.6617422103881836


## 4. Bộ huấn luyện đầy đủ
Ví dụ trên giải thích đồ thị. Bộ huấn luyện dùng cùng họ kiến trúc, khởi tạo chung giữa framework, trọng số lớp cho Retailrocket, clipping và checkpoint. Mã thuật toán dùng cho lượt chính nằm trong src/backends.py và src/training.py.

In [8]:
from src.training import configuration
config,_=configuration('sp500','pytorch')
display(config)

{'seed': 42,
 'hidden_size': 32,
 'batch_size': 256,
 'max_epochs': 20,
 'patience': 4,
 'min_delta': 1e-05,
 'learning_rate': 0.001,
 'adam_epsilon': 1e-07,
 'clip_norm': 1,
 'dataset': 'sp500',
 'framework': 'pytorch',
 'smoke': False,
 'class_weights': [1.0, 1.0, 1.0],
 'metadata_sha256': 'b2d0351f121f2acbb33875db0abe6550dbd5e68c818e63817ddafe7425e023dc',
 'selection': 'validation RMSE'}

## 5. Chủ động chạy cấu hình này
Chỉ chuyển RUN_FULL=True khi muốn huấn luyện đầy đủ. Nên dùng RUN_TRAINING.bat để giảm bộ nhớ của kernel minh họa đang mở.

In [9]:
RUN_FULL=False
if RUN_FULL:
    import subprocess
    environment=os.environ.copy()
    environment.pop('CUDA_VISIBLE_DEVICES',None)
    subprocess.run([sys.executable,'-m','src.training','--dataset','sp500','--framework','pytorch'],cwd=ROOT,env=environment,check=True)
else:
    print('Cell này không khởi động huấn luyện; kết quả đã lưu được đọc ở phần sau.')

Cell này không khởi động huấn luyện; kết quả đã lưu được đọc ở phần sau.


## 6. Dự đoán và kết quả được lưu

In [10]:
folder=ROOT/'results/runs/sp500_pytorch_seed42'
if (folder/'metrics.json').exists():
    display(read_json(folder/'metrics.json'))
    display(pd.read_csv(folder/'predictions.csv').head(12))
else:
    print('Kết quả chính sẽ có sau khi bạn huấn luyện xong.')

{'mae': 0.01001167476717645,
 'rmse': 0.015532880453511007,
 'directional_accuracy': 0.5017104002862947,
 'mean_true_return': 0.0003965526012739019,
 'mean_predicted_return': 0.0007853518937368407,
 'price_mae': 0.9810745793467336,
 'price_rmse': 2.41337032841921,
 'price_reconstruction_clipped': 0,
 'dataset': 'sp500',
 'framework': 'pytorch',
 'seed': 42,
 'best_epoch': 1,
 'completed_epochs': 5,
 'train_seconds': 131.84016350001912,
 'samples': 95007,
 'trainable_parameters': 1313,
 'smoke': False,
 'evaluation_split': 'test'}

,sample_id,timestamp,group,actual,prediction,actual_close,predicted_close
0,1039,1494374400,0,-0.010606,-0.001215,56.27,56.800972
1,1040,1494460800,0,-0.000711,-0.002777,56.23,56.113977
2,1041,1494547200,0,-0.001246,0.000716,56.16,56.270252
3,1042,1494806400,0,0.009569,0.001753,56.70,56.258507
4,1043,1494892800,0,0.002290,-0.002698,56.83,56.547202
5,1044,1494979200,0,-0.018828,0.001172,55.77,56.896642
6,1045,1495065600,0,0.001433,-0.000232,55.85,55.757036
7,1046,1495152000,0,0.005001,0.002632,56.13,55.997178
8,1047,1495411200,0,-0.000891,-0.001501,56.08,56.045791
9,1048,1495497600,0,0.044979,0.002775,58.66,56.235833
